# 08 — Offline Testing

**Tooling** — test steering and scope offline, with no API key.

When you finish you can:

- run an agent with no key and no network, using `TestModel` and `FunctionModel`
- check what the model was allowed to see, and what actually happened (`capture_run_messages`)
- swap a real agent's model and deps inside a test with `agent.override`
- prove no *model* request goes out, with the `ALLOW_MODEL_REQUESTS` kill switch

> This notebook runs **keyless**. CI runs it on every push to `main` and every pull request.

In [ ]:
from learningdantic import setup

MODEL = setup()

## Why test offline

Real models are slow, cost money, and answer differently each time: a poor base for tests.
Pydantic AI ships fake models, so the *plumbing* (tools, deps, validators, history) can be checked
for free, the same way every time.
Docs: [pydantic.dev/docs/ai/guides/testing](https://pydantic.dev/docs/ai/guides/testing/)

## `TestModel` — the keyless fake

By default `TestModel` calls **every** tool the agent has, with placeholder arguments (0 for
numbers), then answers with a JSON summary of the calls. It proves the wiring, not intelligence.
`Agent("test")` is a shortcut for `Agent(TestModel())`; here we keep the `TestModel` in a variable
so we can inspect it in the next cell.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel

test_model = TestModel()
calc_agent = Agent(test_model, instructions="You do arithmetic only with your tools.")


@calc_agent.tool_plain
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@calc_agent.tool_plain
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


r1 = await calc_agent.run("What is 6 times 7?")
print("1. calls every tool:", repr(r1.output))

r2 = await calc_agent.run("What is 6 times 7?", model=TestModel(call_tools=["multiply"]))
print("2. call_tools picks some:", repr(r2.output))

> ✅ **Checkpoint** — `'{"add":0,"multiply":0}'`, then `'{"multiply":0}'`. TestModel ran your
> tools with 0s and recorded the results. `model=` on a run swaps the model for that run only.

## Habit: check what the model saw

Before asking "did it do the right thing?", ask "what was it allowed to see?". `TestModel` keeps
the last request it received, including the tools it was offered. From notebook 09 on, this is how
the course proves an agent's **scope**.

In [ ]:
offered = test_model.last_model_request_parameters.function_tools
for tool in offered:
    print(f"3. {tool.name}: {tool.description}")

assert [tool.name for tool in offered] == ["add", "multiply"]

> ✅ **Checkpoint** — `add: Add two integers.` and `multiply: Multiply two integers.` The docstring
> is what the model reads to choose a tool. Hide a tool (notebook 09) and it vanishes from this list.

## `FunctionModel` — script the model yourself

`TestModel` proves wiring. For behaviour, you write the model's side yourself. A `FunctionModel`
wraps your function: it gets the messages so far and returns the `ModelResponse` you choose,
either a tool call or text.

In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel


def scripted_model(messages, info):
    """First request: call add_numbers(2, 3). Once the tool result is back: answer in text."""
    tool_ran = any(
        isinstance(part, ToolReturnPart) for message in messages for part in message.parts
    )
    if tool_ran:
        return ModelResponse(parts=[TextPart(content="The sum is 5.")])
    return ModelResponse(parts=[ToolCallPart(tool_name="add_numbers", args={"a": 2, "b": 3})])


scripted_agent = Agent(FunctionModel(scripted_model), instructions="Do arithmetic with your tools.")


@scripted_agent.tool_plain
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


r = await scripted_agent.run("what is 2+3?")
print("4. scripted output:", repr(r.output))

> ✅ **Checkpoint** — `'The sum is 5.'`: your function asked for the tool, the real tool ran,
> and your function answered.

## `capture_run_messages` — assert on what happened

A test can check exactly what the agent did: which tools, in which order, with which results.

In [ ]:
from pydantic_ai import capture_run_messages

with capture_run_messages() as messages:
    r = await scripted_agent.run("what is 2+3?")

kinds = [type(part).__name__ for message in messages for part in message.parts]
print("5. captured:", kinds)

assert kinds == ["UserPromptPart", "ToolCallPart", "ToolReturnPart", "TextPart"], kinds
tool_returns = [p for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
assert tool_returns[0].content == 5, tool_returns

> ✅ **Checkpoint** — `['UserPromptPart', 'ToolCallPart', 'ToolReturnPart', 'TextPart']`, and the
> asserts passed: prompt → tool call → tool result (5) → text. A full agent loop, tested in
> milliseconds with no model, no key and no network.

## `override` and the kill switch

Your app's agents use a real model. In a test you swap it out for the length of a `with` block
(`with x:` turns something on for the indented lines, then back off). A keyless `Agent("openai:...")`
fails as soon as it is built (no key), so here we build a real OpenAI model object with a **fake
key**. It never sends anything.

- `agent.override(model=TestModel())` swaps the model inside the `with` block.
- `models.override_allow_model_requests(False)` makes any real *model* request raise before it is
  sent. Fake models are exempt. It blocks model requests only (the part that costs money):
  HTTP calls made by your own tools still go through.

In [ ]:
from pydantic_ai import models
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

real_model = OpenAIResponsesModel("gpt-6-luna", provider=OpenAIProvider(api_key="not-a-real-key"))
live_agent = Agent(real_model, instructions="You are never actually reached.")

with live_agent.override(model=TestModel()):
    r = await live_agent.run("anything")  # answered by TestModel, not OpenAI
print("6. override kept the run offline:", repr(r.output))

with models.override_allow_model_requests(False):
    try:
        await live_agent.run("this would be a real, paid request")
    except RuntimeError as e:
        print("7. blocked:", e)
    else:  # else: runs only if no error was raised
        msg = "a real request went through: the kill switch failed!"
        raise AssertionError(msg)

> ✅ **Checkpoint** — line 6 shows `'success (no tool calls)'` from TestModel. Line 7 shows the
> `RuntimeError` *Model requests are not allowed, since ALLOW_MODEL_REQUESTS is False*, raised
> before anything was sent. Wrap a whole test suite in it and "no paid model calls" is proven,
> not hoped.

## Fake deps with `override(deps=...)`

Notebook 05 gave agents deps (a user, a client, a database). A test wants fake deps, but your app
code passes the real ones deep inside, where the test can't reach. `override(deps=...)` wins over
the deps a run passes, so the test doesn't have to change the app.

In [ ]:
from dataclasses import dataclass

from pydantic_ai import RunContext


@dataclass
class ShopDeps:  # a dataclass: a small class that just holds named fields
    store_name: str


shop_agent = Agent(real_model, deps_type=ShopDeps, instructions="Answer questions about the store.")


@shop_agent.tool
def store_name(ctx: RunContext[ShopDeps]) -> str:
    """Return the name of the store the customer is in."""
    return ctx.deps.store_name


async def app_answer(question: str) -> str:
    """Your app code. It always passes the real deps."""
    result = await shop_agent.run(question, deps=ShopDeps(store_name="Real Store"))
    return result.output


with shop_agent.override(model=TestModel(), deps=ShopDeps(store_name="Fake Test Store")):
    out = await app_answer("Which store am I in?")
print("8. app code ran with fake deps:", repr(out))

assert "Fake Test Store" in out, out

> ✅ **Checkpoint** — `'{"store_name":"Fake Test Store"}'`. The tool read the fake deps even
> though `app_answer` passed `"Real Store"`, and the fake model meant no key was needed.
> (`app_answer` is `async def` because it runs an agent: the async rule.)

## Pin an exact answer

`TestModel(custom_output_text=...)` makes the fake return exactly that text. Handy when the code
after the agent needs a known answer to work on.

In [ ]:
pinned_agent = Agent(TestModel(custom_output_text="EXACT EXPECTED SENTENCE"))

r = await pinned_agent.run("say the thing")
print("9. pinned output:", repr(r.output))

assert r.output == "EXACT EXPECTED SENTENCE"

> ✅ **Checkpoint** — `'EXACT EXPECTED SENTENCE'`, and the assert passed.

## How CI uses this

Open [`.github/workflows/ci.yml`](../.github/workflows/ci.yml). On every push to `main` and every
pull request, GitHub Actions runs `pytest` on this notebook, its solution and the other keyless
notebooks, with **zero secrets**. Green CI means the plumbing works. (A bare `uv run pytest` on your
machine runs every notebook in `notebooks/`, including the live ones that use your key.)

## 🏋️ Exercise — the offline proof

Build an agent on a `FunctionModel` that calls an `add` tool once and then answers (copy
`scripted_model`). Then prove, with plain `assert`s and no API key:

1. **Behaviour:** a spy counter inside `add` shows the tool ran exactly once, and
   `capture_run_messages` shows the `ToolCallPart` came before the `ToolReturnPart`.
2. **Scope, offline:** inside `models.override_allow_model_requests(False)`, run the same agent
   with `model=TestModel()` and assert the model was offered exactly one tool: `add`.

**Stretch:** give the agent a deps dataclass, read it in a tool, and use `override(deps=...)` to
assert your fake was used.

Your take: [`../solutions/08_offline_assertions.ipynb`](../solutions/08_offline_assertions.ipynb).

## What's next

You can now build, watch and test agents. The next six notebooks are about **steering** (what an
agent tries to do) and **scoping** (what it can see, do, spend and remember):

- **09 — Toolsets & MCP** — what it can touch (Scope)
- **10 — Capabilities & guardrails** — steer and fence it (Both)
- **11 — Multi-agent** — one job per agent (Both)
- **12 — Evals** — prove your steering works (Steer)
- **13 — Ship it** — a chat UI, and memory you control (Scope)
- **14 — Capstone** — design a support desk (Both)

## Next →

**[09_toolsets_and_mcp.ipynb](09_toolsets_and_mcp.ipynb)** — control exactly which tools an agent can touch.